# Differential Gene Expression Analysis in Cancer Research

## Author: Jubilee Amechi


## Leukemia Gene Expression Analysis (ALL vs AML)

This notebook implements a basic but complete bioinformatics workflow for analyzing the Golub et al. (1999) leukemia dataset. The goal is to identify differentially expressed genes that separate Acute Lymphoblastic Leukemia (ALL) from Acute Myeloid Leukemia (AML).

The workflow includes:
- Data loading and preprocessing
- Log transformation and normalization
- Variance filtering
- Differential expression analysis (t-test)
- Visualization via clustering heatmaps

## 1. Import Libraries 
I start by importing the required Python libraries.

In [51]:
import numpy as np
import pandas as pd
from scipy import stats
import seaborn as sns
import matplotlib.pyplot as plt

sns.set(style="whitegrid")

## 2. Load Dataset
Assume the dataset is in a tabular format where rows represent genes and columns represent samples.


In [53]:
def load_data(filepath):
    """
    Load gene expression dataset.
    Rows = genes, Columns = samples
    """
    df = pd.read_csv(filepath, index_col=0)
    return df

## 3. Preprocessing
Apply log transformation and also handle missing values in a simple way.

In [73]:
def preprocess_data(df):
    df = df.copy()

    # Replace missing values if any
    df = df.fillna(df.median())

    # Log2 transform (stabilizes variance)
    df = np.log2(df + 1)

    return df

## 4. Labelling
Define sample labels (ALL vs AML). 

In [57]:
labels = None

## 5. Variance Filtering
Remove low-variance genes since they carry little information.


In [59]:
def variance_filter(df, top_percent=0.1):
    variances = df.var(axis=1)
    threshold = np.quantile(variances, 1 - top_percent)
    filtered_df = df.loc[variances >= threshold]
    return filtered_df

## 6. Differential Expression Analysis
Compare gene expression between ALL and AML groups using a two-sample t-test.


In [61]:
def differential_expression(df, labels):
    """
    Returns p-values and fold changes per gene
    """
    all_idx = labels == "ALL"
    aml_idx = labels == "AML"

    all_group = df.loc[:, all_idx]
    aml_group = df.loc[:, aml_idx]

    t_stats, p_values = stats.ttest_ind(all_group, aml_group, axis=1)

    fold_change = aml_group.mean(axis=1) - all_group.mean(axis=1)

    results = pd.DataFrame({
        "t_stat": t_stats,
        "p_value": p_values,
        "log2_fc": fold_change
    }, index=df.index)

    return results.sort_values("p_value")




## 7. Significant Genes Selection
Filter genes based on statistical significance.


In [63]:
def get_significant_genes(results, alpha=0.05):
    return results[results["p_value"] < alpha]

## 8. Visualization (Clustering Heatmap)
I visualize expression patterns of significant genes.


In [65]:
def plot_heatmap(df, labels, title="Gene Expression Heatmap"):
    sns.clustermap(
        df,
        cmap="vlag",
        standard_scale=0
    )
    plt.title(title)
    plt.show()

- This is a simplified statistical pipeline, not a clinical-grade model.
- A real-world extension would include:  Multiple testing correction (FDR), PCA / dimensionality reduction, Machine learning classification (SVM, Random Forest), External validation datasets